# meeting_record_explore — Meeting Record 信件瀏覽（探索用）

輸入時間範圍、BU / SCM、寄件人、標題關鍵字、第幾封到第幾封，把信件內容 print 出來。只讀、不寫表（[c20] 例外：人工刪信）。

- 來源：`BU` = `{catalog}.{schema}.b_internal_meeting_record`、`SCM` = `{catalog}.{schema}.b_internal_meeting_record_scm`
- 時間以 `create_date` 為準，`start_date` ~ `end_date` 兩端都含
- 第幾封 = 過濾後依 `create_date` 排序（`order`：desc 新 → 舊 / asc 舊 → 新）的序號，1 起算
- 用法：改 widget（`source`、`start_date`、`end_date`、`sender`、`from_no`、`to_no`）→ 從 [c10] 往下 Run；catalog / schema / 標題關鍵字 / 排序 / 要印的欄位等直接改 [c01] 內的常數

| cell | 內容 |
|---|---|
| [c10] | 讀表 + 過濾（日期先過濾），印符合筆數與可用欄位 |
| [c11] | 取第 `from_no` ~ `to_no` 封，印目錄（時間、寄件人、標題） |
| [c12] | 逐封印出內容（`COLUMNS` 指定的欄位；空 = 全部欄位） |
| [c20] | 刪除某一封信：在 cell 內填值，先 DRY_RUN 預覽再刪 |

In [ ]:
# [c01] params
# 探索用：只讀、不寫表。改完 widget 後從 [c10] 往下重跑即可。
# 注意：widget 建立過後改預設值不會生效；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。
from datetime import date, datetime, timedelta

dbutils.widgets.dropdown("source", "ALL", ["ALL", "BU", "SCM"])
dbutils.widgets.text("start_date", "")  # yyyy-MM-dd；空 = end_date 往前 6 天（共 7 天）
dbutils.widgets.text("end_date", "")    # yyyy-MM-dd，含當天；空 = 今天
dbutils.widgets.text("sender", "")      # mail_from 包含此字串（不分大小寫）；空 = 全部
dbutils.widgets.text("from_no", "1")    # 從第幾封（1 起算，含）
dbutils.widgets.text("to_no", "5")      # 到第幾封（含）

# 不常改的設定直接寫在這裡
CATALOG = "micenter"
SCHEMA = "mi3_datahub_prod"
SUBJECT = ""       # mail_subject 包含此字串（不分大小寫）；空 = 全部
ORDER = "desc"     # 依 create_date；desc = 新 → 舊、asc = 舊 → 新
COLUMNS = []       # 內容要印的欄位，例如 ["payload"]；空 = 全部欄位
MAX_CHARS = 3000   # 每個欄位最多印幾個字；0 = 不截斷
MAX_MAILS = 200    # 一次最多印幾封，避免把大量內文拉回 driver


def _date(s: str) -> date | None:
    s = s.strip()
    return datetime.strptime(s, "%Y-%m-%d").date() if s else None


_end = _date(dbutils.widgets.get("end_date")) or date.today()
cfg = {
    "catalog": CATALOG,
    "schema": SCHEMA,
    "source": dbutils.widgets.get("source"),
    "start_date": _date(dbutils.widgets.get("start_date")) or _end - timedelta(days=6),
    "end_date": _end,
    "sender": dbutils.widgets.get("sender").strip(),
    "subject": SUBJECT.strip(),
    "order": ORDER,
    "from_no": int(dbutils.widgets.get("from_no")),
    "to_no": int(dbutils.widgets.get("to_no")),
    "columns": COLUMNS,
    "max_chars": MAX_CHARS,
}
assert cfg["start_date"] <= cfg["end_date"], "start_date 不可晚於 end_date"
assert cfg["order"] in ("desc", "asc"), "ORDER 只能是 desc / asc"
assert 1 <= cfg["from_no"] <= cfg["to_no"], "需 1 <= from_no <= to_no"
assert cfg["to_no"] - cfg["from_no"] + 1 <= MAX_MAILS, f"一次最多 {MAX_MAILS} 封，請縮小範圍"
print(cfg)


In [ ]:
# [c02] imports
# 本 cell 與 [c03] 不碰 spark / dbutils，可被 tests/ 載入。
import json
from datetime import date, timedelta
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

SOURCES = {"BU": "b_internal_meeting_record", "SCM": "b_internal_meeting_record_scm"}
HEADER_COLS = ["source", "create_date", "mail_from", "mail_subject"]

In [ ]:
# [c03] pure_helpers
def resolve_sources(source: str) -> dict[str, str]:
    """'ALL' / 'BU' / 'SCM' → {標籤: 表名}。"""
    key = source.strip().upper()
    if key == "ALL":
        return dict(SOURCES)
    if key not in SOURCES:
        raise ValueError(f"source 只能是 ALL / {' / '.join(SOURCES)}，收到 {source!r}")
    return {key: SOURCES[key]}


def filter_mail(df: DataFrame, sender: str = "", subject: str = "") -> DataFrame:
    """mail_from / mail_subject 包含關鍵字（不分大小寫）；空字串 = 不過濾。"""
    for col, kw in (("mail_from", sender), ("mail_subject", subject)):
        kw = kw.strip().lower()
        if kw:
            df = df.where(F.lower(F.col(col)).contains(kw))
    return df


def take_range(df: DataFrame, order: str, from_no: int, to_no: int) -> DataFrame:
    """依 create_date 排序後取第 from_no ~ to_no 封（1 起算、含頭尾）。

    同一時間的信再依 source / mail_from / mail_subject 排，讓序號每次跑都一樣。
    """
    ts = F.col("create_date").desc() if order == "desc" else F.col("create_date").asc()
    return (
        df.orderBy(ts, "source", "mail_from", "mail_subject")
        .limit(to_no)
        .offset(from_no - 1)
    )


def _to_text(v, max_chars: int) -> str:
    """欄位值轉成可讀字串：JSON 字串 / dict / list 排版；超過 max_chars 截斷（0 = 不截斷）。"""
    if v is None:
        return "(null)"
    if isinstance(v, str):
        s = v.strip()
        if s[:1] in ("{", "["):
            try:
                s = json.dumps(json.loads(s), ensure_ascii=False, indent=2)
            except ValueError:
                pass
    elif isinstance(v, (dict, list)):
        s = json.dumps(v, ensure_ascii=False, indent=2, default=str)
    else:
        s = str(v)
    if max_chars and len(s) > max_chars:
        s = s[:max_chars] + f"\n…（截斷，共 {len(s)} 字）"
    return s


def format_index(no: int, row: dict) -> str:
    """目錄一行：序號、來源、時間、寄件人、標題。"""
    head = f"#{no:<4} [{row.get('source')}] {row.get('create_date')}"
    return f"{head}  {row.get('mail_from')}  |  {row.get('mail_subject')}"


def format_mail(no: int, row: dict, columns: list[str], max_chars: int) -> str:
    """一封信的完整內容。columns 空 = 印 HEADER_COLS 以外的全部欄位。"""
    body_cols = columns or [c for c in row if c not in HEADER_COLS]
    lines = [
        "=" * 100,
        f"#{no}  [{row.get('source')}]  {row.get('create_date')}",
        f"From   : {row.get('mail_from')}",
        f"Subject: {row.get('mail_subject')}",
    ]
    for c in body_cols:
        lines.append(f"---- {c} ----")
        lines.append(_to_text(row[c], max_chars) if c in row else "(無此欄位)")
    return "\n".join(lines)

In [ ]:
# [c10] load_filter
# 先用日期過濾再做關鍵字過濾；範圍 [start_date, end_date + 1)。
srcs = resolve_sources(cfg["source"])
start_ts, end_ts = cfg["start_date"], cfg["end_date"] + timedelta(days=1)
parts = [
    spark.table(f"{cfg['catalog']}.{cfg['schema']}.{t}")
    .where((F.col("create_date") >= F.lit(start_ts)) & (F.col("create_date") < F.lit(end_ts)))
    .withColumn("source", F.lit(label))
    for label, t in srcs.items()
]
# BU / SCM 兩張表欄位可能不完全一樣，缺的欄位補 null
df_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parts)
df_hit = filter_mail(df_all, cfg["sender"], cfg["subject"])
# count 只跑在日期過濾後的小範圍，探索用；範圍拉很長時這一步會比較慢
n_hit = df_hit.count()
print(f"range: {cfg['start_date']} ~ {cfg['end_date']}, source={list(srcs)}, "
      f"sender={cfg['sender']!r}, subject={cfg['subject']!r} → 符合 {n_hit} 封")
print("可用欄位（填到 [c01] COLUMNS）:", [c for c in df_hit.columns if c not in HEADER_COLS])

In [ ]:
# [c11] index
# 只 collect 第 from_no ~ to_no 封（最多 MAX_MAILS 封）。
mail_rows = [
    r.asDict(recursive=True)
    for r in take_range(df_hit, cfg["order"], cfg["from_no"], cfg["to_no"]).collect()
]
mail_nos = range(cfg["from_no"], cfg["from_no"] + len(mail_rows))
if not mail_rows:
    print(f"沒有信：符合 {n_hit} 封，from_no={cfg['from_no']} 已超過")
else:
    print(f"第 {mail_nos[0]} ~ {mail_nos[-1]} 封 / 共 {n_hit} 封（order={cfg['order']}）")
    for no, row in zip(mail_nos, mail_rows, strict=True):
        print(format_index(no, row))

In [ ]:
# [c12] print_mail
for no, row in zip(mail_nos, mail_rows, strict=True):
    print(format_mail(no, row, cfg["columns"], cfg["max_chars"]))

In [ ]:
# [c20] delete_mail
# ⚠ 會真的刪 Delta 表資料。刻意不用 widget：直接改下面的值。
# 流程：先 DRY_RUN = True 跑一次看預覽 → 確認是要刪的那封 → 改 DRY_RUN = False 再跑。
# b_ 表原則上 append-only，這裡是人工清錯信。
# 刪錯可用 time travel 還原（刪除後會印出 RESTORE 指令）。
from delta.tables import DeltaTable

# ---- 在這裡填要刪的信 ----
NO = None  # 填 [c11] 目錄的序號（例如 3）就直接用那封的值，下面 4 個不用填；用手填就留 None
DEL_SOURCE = "BU"  # "BU" / "SCM"
DEL_CREATE_DATE = "2026-09-24 09:00:00"  # 照 [c11] 印出的時間貼上（有微秒要一起貼）
DEL_MAIL_FROM = "Amy <amy@corp.com>"  # 原文完整字串，不是關鍵字
DEL_MAIL_SUBJECT = "專案會議"  # 原文完整字串；標題是 null 就填 None
EXPECTED_ROWS = 1  # 預期刪幾筆；實際符合筆數不同就中止
DRY_RUN = True
# -------------------------

if NO is not None:
    _i = NO - cfg["from_no"]
    assert 0 <= _i < len(mail_rows), f"NO={NO} 不在目前 [c11] 印出的序號範圍內"
    _row = mail_rows[_i]
    del_key = {k: _row[k] for k in ("source", "create_date", "mail_from", "mail_subject")}
    del_ts = F.lit(del_key["create_date"])  # collect 回來的 datetime，精確到微秒
else:
    del_key = {"source": DEL_SOURCE, "create_date": DEL_CREATE_DATE,
               "mail_from": DEL_MAIL_FROM, "mail_subject": DEL_MAIL_SUBJECT}
    del_ts = F.to_timestamp(F.lit(DEL_CREATE_DATE), "yyyy-MM-dd HH:mm:ss[.SSSSSS]")

(del_tbl,) = resolve_sources(del_key["source"]).values()
del_table = f"{cfg['catalog']}.{cfg['schema']}.{del_tbl}"
# 4 個欄位全部比對；eqNullSafe 讓 null 標題 / 寄件人也能精確對到
del_cond = (
    (F.col("create_date") == del_ts)
    & F.col("mail_from").eqNullSafe(F.lit(del_key["mail_from"]))
    & F.col("mail_subject").eqNullSafe(F.lit(del_key["mail_subject"]))
)

df_del = spark.table(del_table).where(del_cond)
preview = [r.asDict(recursive=True) for r in df_del.limit(EXPECTED_ROWS + 10).collect()]
print(f"table: {del_table}")
print(f"key  : {del_key}")
print(f"符合 {len(preview)} 筆（預期 {EXPECTED_ROWS}）:")
for _no, _r in enumerate(preview, 1):
    print("  " + format_index(_no, {**_r, "source": del_key["source"]}))

if len(preview) != EXPECTED_ROWS:
    print("✋ 筆數和 EXPECTED_ROWS 不符，不刪除。時間 / 寄件人 / 標題要和原文完全一樣。")
elif DRY_RUN:
    print("DRY_RUN = True：只預覽，沒有刪除。確認無誤後改 DRY_RUN = False 再跑本 cell。")
else:
    dt = DeltaTable.forName(spark, del_table)
    dt.delete(del_cond)
    h = dt.history(1).select("version", "operation", "operationMetrics").first()
    print(f"已刪除：version={h['version']}, {h['operation']}, "
          f"numDeletedRows={h['operationMetrics'].get('numDeletedRows')}")
    print(f"刪錯要還原：RESTORE TABLE {del_table} TO VERSION AS OF {h['version'] - 1}")
